# LeetCode 3688 — Bitwise OR of Even Numbers in an Array bằng Python

Link đề: https://leetcode.com/problems/bitwise-or-of-even-numbers-in-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation, Simulation

Skeleton LeetCode:

```python
class Solution:
    def evenNumberBitwiseORs(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lấy **phép OR bit (`|`)** của **tất cả các số chẵn** trong mảng; nếu không có số chẵn nào thì trả `0`.
2. Nắm phép **OR bit**: bit kết quả là `1` nếu **ít nhất một** toán hạng có bit đó bằng `1`; và biết `0` là **phần tử trung hòa** của OR (`x | 0 = x`).
3. Biết kiểm tra số chẵn bằng `x % 2 == 0` hoặc bằng bit thấp nhất `x & 1 == 0`.
4. Viết 4 cách (gom số chẵn rồi OR, một lượt cộng dồn `|=`, `functools.reduce` trên `filter`, xét từng **cột bit**) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: OR cả số lẻ, dùng `and`/`or` **logic** thay vì `&`/`|` **bit**, khởi tạo bằng `nums[0]`, và dùng `+` thay cho `|`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về **OR bit** của **tất cả các số chẵn** trong mảng. Nếu mảng **không có số chẵn nào**, trả về `0`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,3,4,5,6]` | `6` | các số chẵn là `2, 4, 6`; `2 \| 4 \| 6 = 6` |
| 2 | `[7,9,11]` | `0` | không có số chẵn nào nên kết quả là `0` |
| 3 | `[1,8,16]` | `24` | các số chẵn là `8, 16`; `8 \| 16 = 24` |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`.

In [1]:
nums = [1, 8, 16]
evens = [x for x in nums if x % 2 == 0]
print("các số chẵn:", evens)
result = 0
for x in evens:
    result |= x
print("OR =", result, "| nhị phân:", bin(result))
print(2 | 4 | 6, 7 | 0, 8 | 16)

các số chẵn: [8, 16]
OR = 24 | nhị phân: 0b11000
6 7 24


## 3. Kiến thức nền

### 3.1. Số chẵn / lẻ

`x % 2 == 0` -> chẵn; `x % 2 == 1` -> lẻ. Với số nguyên dương, `x` chẵn **khi và chỉ khi bit thấp nhất (bit 0) bằng `0`**: `x & 1 == 0`. Hai cách tương đương; dùng cách nào quen tay hơn.

In [2]:
for x in (1, 2, 3, 4, 100):
    print(x, bin(x), "chẵn" if x % 2 == 0 else "lẻ", "| bit0 =", x & 1)

1 0b1 lẻ | bit0 = 1
2 0b10 chẵn | bit0 = 0
3 0b11 lẻ | bit0 = 1
4 0b100 chẵn | bit0 = 0
100 0b1100100 chẵn | bit0 = 0


### 3.2. Biểu diễn nhị phân và `bin()`

Mỗi số nguyên là tổ hợp của các lũy thừa của 2. `bin(6) = '0b110'` nghĩa là `4 + 2`. Bit thứ `i` (đếm từ `0` bên phải) có trọng số `2^i`. Muốn xem bit thứ `i` của `x`: `(x >> i) & 1`.

In [3]:
print(bin(6), bin(8), bin(16), bin(24))
x = 24
print([ (x >> i) & 1 for i in range(6) ])        # bit 0..5 của 24 = 0b11000

0b110 0b1000 0b10000 0b11000
[0, 0, 0, 1, 1, 0]


### 3.3. Phép OR bit `|`

`a | b` so sánh từng cặp bit: kết quả `1` nếu **một trong hai** (hoặc cả hai) bằng `1`. Bảng: `0|0=0`, `0|1=1`, `1|0=1`, `1|1=1`. Tính chất: **giao hoán**, **kết hợp**, **lũy đẳng** (`a | a = a`), và **`0` là phần tử trung hòa** (`a | 0 = a`). Nhờ đó ta có thể cộng dồn `res |= x` theo bất kỳ thứ tự nào, khởi tạo `res = 0`.

In [4]:
a, b = 0b0110, 0b1010
print(bin(a | b), bin(a & b), bin(a ^ b))
print(5 | 5, 5 | 0, (3 | 5) | 6 == 3 | (5 | 6))

0b1110 0b10 0b1100
5 5 True


### 3.4. `|` (bit) khác `or` (logic)

`or` là toán tử **logic**: trả về toán hạng đầu nếu nó "truthy", ngược lại trả toán hạng sau (`8 or 16` cho `8`). `|` mới là OR **bit** (`8 | 16 = 24`). Hai thứ hoàn toàn khác nhau, đừng lẫn.

In [5]:
print(8 or 16, 8 | 16)
print(0 or 16, 0 | 16)

8 24
16 16


### 3.5. `functools.reduce` và `operator.or_`

`reduce(f, iterable, init)` gấp dãy bằng hàm hai biến: `f(f(f(init, a1), a2), a3)`. Với OR: `reduce(or_, evens, 0)`; giá trị khởi tạo `0` đảm bảo dãy rỗng cho kết quả `0` (đúng đề).

In [6]:
from functools import reduce
from operator import or_
print(reduce(or_, [2, 4, 6], 0), reduce(or_, [], 0))

6 0


### 3.6. So sánh nhanh ba phép bit: OR, AND, XOR

| a | b | `a \| b` | `a & b` | `a ^ b` |
|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 1 | 0 | 1 |
| 1 | 0 | 1 | 0 | 1 |
| 1 | 1 | 1 | 1 | 0 |

OR "bật" bit nếu **có ít nhất một** số bật; AND chỉ bật nếu **tất cả** cùng bật; XOR bật nếu số lượng số bật là **lẻ**. Bài này dùng OR nên kết quả luôn là "hợp" (union) các bit của các số chẵn: kết quả **không bao giờ nhỏ hơn** từng số chẵn thành phần.

In [7]:
a, b = 12, 10
print(f"{a:04b} | {b:04b} = {a | b:04b}")
print(f"{a:04b} & {b:04b} = {a & b:04b}")
print(f"{a:04b} ^ {b:04b} = {a ^ b:04b}")
print(all((a | b) >= v for v in (a, b)))

1100 | 1010 = 1110
1100 & 1010 = 1000
1100 ^ 1010 = 0110
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay `nums = [1, 2, 3, 4, 5, 6]`:

```
số chẵn: 2 = 010, 4 = 100, 6 = 110
OR từng cột bit:
  bit2: 0 | 1 | 1 = 1
  bit1: 1 | 0 | 1 = 1
  bit0: 0 | 0 | 0 = 0
kết quả = 110 = 6
```

Làm tay `[1, 8, 16]`: chẵn `8 = 01000` và `16 = 10000` -> OR `11000 = 24`. Làm tay `[7, 9, 11]`: không có số chẵn -> `0`.

Hai nhận xét:

1. Mỗi số chẵn có bit 0 bằng `0` nên kết quả OR **luôn chẵn**.
2. Vì `nums[i] <= 100 < 128`, kết quả luôn `<= 127` (7 bit).

## 5. Cách 1 — Brute Force / baseline

Làm đúng hai bước của đề: (1) **lọc** các số chẵn vào danh sách `evens`; (2) **OR** chúng lại, bắt đầu từ `0`.

Nút thắt: tạo thêm danh sách trung gian (bộ nhớ `O(n)`) và hai lượt duyệt. Không sai gì nhưng ta có thể gộp thành một lượt.

In [8]:
class SolutionBrute:
    def evenNumberBitwiseORs(self, nums):
        evens = [x for x in nums if x % 2 == 0]       # bước 1: lọc
        result = 0
        for x in evens:                               # bước 2: OR
            result |= x
        return result

print(SolutionBrute().evenNumberBitwiseORs([1, 2, 3, 4, 5, 6]))
print(SolutionBrute().evenNumberBitwiseORs([7, 9, 11]))
print(SolutionBrute().evenNumberBitwiseORs([1, 8, 16]))

6
0
24


**Độ phức tạp:** `n = len(nums)`. Time `O(n)`, Space `O(n)` cho danh sách `evens`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần danh sách trung gian. Vì `0` là phần tử trung hòa của OR và OR giao hoán/kết hợp, ta cộng dồn **trực tiếp** trong một lượt:

```
result = 0
for x in nums:
    if x là số chẵn: result |= x
```

**Quy tắc "nếu… thì…":**

- Nếu `x % 2 == 0` -> `result |= x`.
- Nếu `x` lẻ -> bỏ qua.
- Nếu không có số chẵn nào -> `result` giữ nguyên giá trị khởi tạo `0` (đúng đề).

Invariant: sau khi xử lý `i` phần tử đầu, `result` là OR của các số chẵn trong `i` phần tử đó.

In [9]:
def even_or(nums):
    result = 0
    for x in nums:
        if x % 2 == 0:
            result |= x
    return result

print(even_or([1, 2, 3, 4, 5, 6]), even_or([7, 9, 11]), even_or([1, 8, 16]))

6 0 24


## 7. Các cách tiếp theo

### Cách 2 — Một lượt cộng dồn `|=` (cách nộp bài)

Chính là ý tưởng mục 6: không danh sách phụ, `O(1)` bộ nhớ.

In [10]:
class SolutionLoop:
    def evenNumberBitwiseORs(self, nums):
        result = 0
        for x in nums:
            if x % 2 == 0:
                result |= x
        return result

print(SolutionLoop().evenNumberBitwiseORs([1, 2, 3, 4, 5, 6]), SolutionLoop().evenNumberBitwiseORs([1, 8, 16]))

6 24


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — `reduce` trên `filter`

Phong cách hàm: `filter` giữ số chẵn, `reduce(or_, ..., 0)` gấp bằng OR. Gọn một dòng, cho thấy "OR cả dãy" chính là một phép **gấp (fold)** với phần tử trung hòa `0`.

In [11]:
from functools import reduce
from operator import or_

class SolutionReduce:
    def evenNumberBitwiseORs(self, nums):
        return reduce(or_, filter(lambda x: x % 2 == 0, nums), 0)

print(SolutionReduce().evenNumberBitwiseORs([1, 2, 3, 4, 5, 6]), SolutionReduce().evenNumberBitwiseORs([7, 9, 11]))

6 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (iterator, không tạo danh sách).

### Cách 4 — Xét từng cột bit

Thay vì cộng dồn, ta hỏi cho **mỗi bit `b` (0..6)**: "có số **chẵn** nào có bit `b` bằng `1` không?". Nếu có thì bật bit `b` của kết quả. Đây là cách nhìn "OR = với mỗi cột, ít nhất một số có 1", hữu ích khi hiểu AND/OR theo cột.

In [12]:
class SolutionBitColumns:
    def evenNumberBitwiseORs(self, nums):
        evens = [x for x in nums if x % 2 == 0]
        result = 0
        for b in range(7):                              # 100 < 128 = 2^7 -> đủ 7 bit
            if any((x >> b) & 1 for x in evens):
                result += 1 << b                        # bật bit b (mỗi bit chỉ cộng một lần)
        return result

print(SolutionBitColumns().evenNumberBitwiseORs([1, 2, 3, 4, 5, 6]), SolutionBitColumns().evenNumberBitwiseORs([1, 8, 16]))

6 24


**Độ phức tạp:** Time `O(7 * n) = O(n)` (số bit là hằng số), Space `O(n)` cho `evens` (có thể dùng generator để còn `O(1)`).

## 8. TRACE từng bước

Trace `nums = [1, 8, 16]` với `result` ban đầu `0`.

| x | chẵn? | result trước (nhị phân) | x (nhị phân) | result sau |
|---|---|---|---|---|
| 1 | không | `00000` | - | `00000` = 0 |
| 8 | có | `00000` | `01000` | `01000` = 8 |
| 16 | có | `01000` | `10000` | `11000` = 24 |

In [13]:
def trace_or(nums):
    print("nums =", nums)
    result = 0
    for x in nums:
        if x % 2 == 0:
            before = result
            result |= x
            print(f"x={x:>3} ({x:07b}) chẵn -> {before:07b} | {x:07b} = {result:07b} ({result})")
        else:
            print(f"x={x:>3} ({x:07b}) lẻ   -> bỏ qua, result={result:07b} ({result})")
    print("kết quả:", result)
    return result

trace_or([1, 8, 16])
trace_or([1, 2, 3, 4, 5, 6])
trace_or([7, 9, 11])

nums = [1, 8, 16]
x=  1 (0000001) lẻ   -> bỏ qua, result=0000000 (0)
x=  8 (0001000) chẵn -> 0000000 | 0001000 = 0001000 (8)
x= 16 (0010000) chẵn -> 0001000 | 0010000 = 0011000 (24)
kết quả: 24
nums = [1, 2, 3, 4, 5, 6]
x=  1 (0000001) lẻ   -> bỏ qua, result=0000000 (0)
x=  2 (0000010) chẵn -> 0000000 | 0000010 = 0000010 (2)
x=  3 (0000011) lẻ   -> bỏ qua, result=0000010 (2)
x=  4 (0000100) chẵn -> 0000010 | 0000100 = 0000110 (6)
x=  5 (0000101) lẻ   -> bỏ qua, result=0000110 (6)
x=  6 (0000110) chẵn -> 0000110 | 0000110 = 0000110 (6)
kết quả: 6
nums = [7, 9, 11]
x=  7 (0000111) lẻ   -> bỏ qua, result=0000000 (0)
x=  9 (0001001) lẻ   -> bỏ qua, result=0000000 (0)
x= 11 (0001011) lẻ   -> bỏ qua, result=0000000 (0)
kết quả: 0


0

### Trace theo cột bit

Với `[1, 2, 3, 4, 5, 6]`, các số chẵn là `2, 4, 6`. Bảng cột bit (bit 2, 1, 0):

| số chẵn | bit2 | bit1 | bit0 |
|---|---|---|---|
| 2 | 0 | 1 | 0 |
| 4 | 1 | 0 | 0 |
| 6 | 1 | 1 | 0 |
| **OR cột** | 1 | 1 | 0 |

Đọc hàng cuối: `110` = `6`.

In [14]:
def trace_columns(nums, bits=7):
    evens = [x for x in nums if x % 2 == 0]
    print("số chẵn:", evens)
    result = 0
    for b in range(bits - 1, -1, -1):
        col = [(x >> b) & 1 for x in evens]
        bit = 1 if any(col) else 0
        result |= bit << b
        if any(col):
            print(f"bit {b}: cột={col} -> OR={bit}")
    print("kết quả:", result, bin(result))
    return result

assert trace_columns([1, 2, 3, 4, 5, 6]) == 6
assert trace_columns([1, 8, 16]) == 24

số chẵn: [2, 4, 6]
bit 2: cột=[0, 1, 1] -> OR=1
bit 1: cột=[1, 0, 1] -> OR=1
kết quả: 6 0b110
số chẵn: [8, 16]
bit 4: cột=[0, 1] -> OR=1
bit 3: cột=[1, 0] -> OR=1
kết quả: 24 0b11000


## 9. Code nộp LeetCode

In [15]:
from typing import List, Optional

class Solution:
    def evenNumberBitwiseORs(self, nums: List[int]) -> int:
        result = 0
        for x in nums:
            if x % 2 == 0:
                result |= x
        return result

## 10. Vì sao thuật toán đúng

- **Khởi tạo:** `0` là phần tử trung hòa của OR (`0 | a = a`), nên OR của một tập rỗng được quy ước là `0`, đúng với yêu cầu "không có số chẵn thì trả `0`".
- **Cộng dồn:** OR có tính **kết hợp và giao hoán**, nên `((0 | a1) | a2) | ... | ak` bằng OR của tập `{a1, ..., ak}` bất kể thứ tự duyệt. Lũy đẳng (`a | a = a`) nghĩa là số lặp lại không ảnh hưởng.
- **Chọn đúng số chẵn:** điều kiện `x % 2 == 0` giữ đúng các số chẵn và bỏ qua số lẻ; số lẻ **không** đóng góp vào kết quả vì ta không OR chúng.
- **Tính chất kiểm tra:** mọi số chẵn có bit 0 bằng `0`, nên kết quả luôn chẵn; kết quả là `0` khi và chỉ khi không có số chẵn nào (vì `nums[i] >= 1` nên mỗi số chẵn `>= 2` làm kết quả `> 0`).

## 11. Phân tích Time / Space Complexity

`n = len(nums) <= 100`.

| Cách | Time | Space |
|---|---|---|
| Lọc rồi OR (hai lượt) | `O(n)` | `O(n)` |
| Một lượt `\|=` | `O(n)` | `O(1)` |
| `reduce` + `filter` | `O(n)` | `O(1)` |
| Xét từng cột bit | `O(n * B)`, `B = 7` | `O(n)` hoặc `O(1)` |

Mọi cách đều tuyến tính; cách một lượt là gọn và tiết kiệm nhất.

## 12. Bộ test

Ví dụ của đề, các biên (không có số chẵn, toàn số chẵn, một phần tử, giá trị lớn nhất `100`) và đối chiếu **tất cả cách**.

In [16]:
impls = [SolutionBrute(), SolutionLoop(), SolutionReduce(), SolutionBitColumns(), Solution()]

cases = [
    ([1, 2, 3, 4, 5, 6], 6),
    ([7, 9, 11], 0),
    ([1, 8, 16], 24),
    ([2], 2),                         # n = 1, chẵn
    ([1], 0),                         # n = 1, lẻ
    ([100], 100),                     # giá trị lớn nhất
    ([100, 98, 2], 100 | 98 | 2),     # 1100100 | 1100010 | 0000010 = 1100110 = 102
    ([2, 2, 2], 2),                   # lặp lại không đổi
    ([64, 32, 3], 96),
    ([99, 100], 100),                 # lẻ lớn bị bỏ
]
for nums, exp in cases:
    assert 1 <= len(nums) <= 100 and all(1 <= v <= 100 for v in nums)
    for impl in impls:
        got = impl.evenNumberBitwiseORs(nums[:])
        assert got == exp, (type(impl).__name__, nums, got, exp)
print("ví dụ + edge OK:", len(cases), "ca;", "100|98|2 =", 100 | 98 | 2)

ví dụ + edge OK: 10 ca; 100|98|2 = 102


In [17]:
import random
random.seed(3688)

def ref(nums):
    out = 0
    for x in nums:
        if not (x & 1):                     # kiểm tra chẵn bằng bit thay vì %
            out = out | x
    return out

for _ in range(3000):
    n = random.randint(1, 100)
    nums = [random.randint(1, 100) for _ in range(n)]
    exp = ref(nums)
    for impl in impls:
        assert impl.evenNumberBitwiseORs(nums[:]) == exp, (nums[:6], type(impl).__name__)
    assert exp % 2 == 0 and 0 <= exp <= 127           # luôn chẵn và < 128
    assert (exp == 0) == all(x % 2 == 1 for x in nums)  # 0 khi và chỉ khi không có số chẵn
print("3000 test ngẫu nhiên (seed=3688) OK")

3000 test ngẫu nhiên (seed=3688) OK


In [18]:
# Tính chất: OR không đổi khi đảo thứ tự hoặc lặp lại phần tử
for _ in range(500):
    nums = [random.randint(1, 100) for _ in range(random.randint(1, 30))]
    shuffled = nums[:]
    random.shuffle(shuffled)
    doubled = nums + nums
    base = Solution().evenNumberBitwiseORs(nums)
    assert Solution().evenNumberBitwiseORs(shuffled) == base
    assert Solution().evenNumberBitwiseORs(doubled) == base
print("bất biến theo thứ tự và theo lặp lại: OK")

bất biến theo thứ tự và theo lặp lại: OK


In [19]:
# Kiểm tra tính đơn điệu: thêm một số chẵn vào mảng không bao giờ làm kết quả giảm
for _ in range(500):
    nums = [random.randint(1, 100) for _ in range(random.randint(1, 50))]
    extra = random.choice(range(2, 101, 2))
    base = Solution().evenNumberBitwiseORs(nums)
    more = Solution().evenNumberBitwiseORs(nums + [extra])
    assert more >= base and (more | base) == more and (more | extra) == more
# Thêm một số lẻ thì kết quả không đổi
for _ in range(500):
    nums = [random.randint(1, 100) for _ in range(random.randint(1, 50))]
    odd = random.choice(range(1, 101, 2))
    assert Solution().evenNumberBitwiseORs(nums + [odd]) == Solution().evenNumberBitwiseORs(nums)
print("thêm số chẵn: kết quả không giảm; thêm số lẻ: kết quả không đổi")

thêm số chẵn: kết quả không giảm; thêm số lẻ: kết quả không đổi


## 13. Các lỗi người mới hay gặp

### Lỗi 1: OR cả số lẻ

Quên lọc số chẵn. Với `[1,2,3,4,5,6]` ta được `7` thay vì `6`.

In [20]:
def wrong_all(nums):
    result = 0
    for x in nums:
        result |= x                    # SAI: không lọc chẵn
    return result

nums = [1, 2, 3, 4, 5, 6]
print(wrong_all(nums), "!=", Solution().evenNumberBitwiseORs(nums))
assert wrong_all(nums) != Solution().evenNumberBitwiseORs(nums)      # 7 != 6

7 != 6


### Lỗi 2: dùng `or` logic thay cho `|` bit

`res = res or x` giữ giá trị đầu tiên "truthy" chứ không gộp bit.

In [21]:
def wrong_logical_or(nums):
    result = 0
    for x in nums:
        if x % 2 == 0:
            result = result or x       # SAI: toán tử logic
    return result

nums = [1, 8, 16]
print(wrong_logical_or(nums), "!=", Solution().evenNumberBitwiseORs(nums))
assert wrong_logical_or(nums) != Solution().evenNumberBitwiseORs(nums)   # 8 != 24

8 != 24


### Lỗi 3: khởi tạo bằng `nums[0]`

Nếu `nums[0]` là số **lẻ** thì nó bị OR vào kết quả. Phải khởi tạo `0` (phần tử trung hòa).

In [22]:
def wrong_init(nums):
    result = nums[0]                   # SAI: phần tử đầu có thể lẻ
    for x in nums:
        if x % 2 == 0:
            result |= x
    return result

nums = [1, 8, 16]
print(wrong_init(nums), "!=", Solution().evenNumberBitwiseORs(nums))
assert wrong_init(nums) != Solution().evenNumberBitwiseORs(nums)         # 25 != 24
assert wrong_init([7, 9, 11]) != Solution().evenNumberBitwiseORs([7, 9, 11])  # 7 != 0

25 != 24


### Lỗi 4: cộng (`+`) thay vì OR

`+` chỉ trùng OR khi các số **không chung bit nào**. Với `[2,4,6]`, `2 + 4 + 6 = 12` nhưng `2 | 4 | 6 = 6` (bit chung bị cộng thừa).

In [23]:
def wrong_sum(nums):
    return sum(x for x in nums if x % 2 == 0)

nums = [1, 2, 3, 4, 5, 6]
print(wrong_sum(nums), "!=", Solution().evenNumberBitwiseORs(nums))
assert wrong_sum(nums) != Solution().evenNumberBitwiseORs(nums)          # 12 != 6

12 != 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "OR / AND / XOR của một nhóm phần tử" -> **một biến cộng dồn** `res |= x` (hoặc `&=`, `^=`).
- Chọn **phần tử trung hòa** đúng: OR và XOR khởi tạo `0`; AND khởi tạo toàn bit `1` (ví dụ `-1`).
- Nhớ `|`, `&`, `^` là **bit**; `or`, `and`, `not` là **logic**.
- Chẵn/lẻ: `x % 2 == 0` hoặc `x & 1 == 0`. Số chẵn luôn có bit 0 bằng `0`.
- OR **lũy đẳng** (`a | a = a`) nên phần tử lặp không cần quan tâm; OR chỉ **tăng** (không bao giờ bớt bit).
- Hiểu theo **cột bit** giúp kiểm tra: mỗi bit của kết quả là OR của cột đó.

## 15. Biến thể của bài

1. **OR của các số lẻ** (đổi điều kiện lọc).
2. **AND của các số chẵn** (khởi tạo bằng `-1`/toàn bit 1, cẩn thận khi không có số chẵn).
3. **XOR của các số chẵn** (khởi tạo `0`).
4. **Single Number** (LeetCode 136): XOR cả mảng để tìm số xuất hiện một lần.
5. **Bitwise OR of all subarrays / Number of Subarrays with OR** (LeetCode 898, 2044...): mở rộng sang mảng con.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Lọc rồi OR | `evens` rồi `for` OR | `O(n)` | `O(n)` | Đúng từng bước đề, dễ debug |
| Một lượt `\|=` | `if chẵn: res \|= x` | `O(n)` | `O(1)` | Cách nộp chuẩn |
| `reduce` + `filter` | gấp bằng `or_` từ `0` | `O(n)` | `O(1)` | Phong cách hàm, một dòng |
| Từng cột bit | mỗi bit: có số chẵn nào bật? | `O(7 n)` | `O(n)` | Để hiểu OR theo cột bit |

Nộp cách một lượt; dùng cách cột bit để hiểu bản chất.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().evenNumberBitwiseORs([3, 5, 12, 10])` là gì?

**Bài 2 (tính tay):** `[20, 6, 9]`: viết nhị phân các số chẵn và OR theo cột.

**Bài 3 (điền TODO):** hoàn thành `odd_or(nums)` trả OR của các số **lẻ**.

**Bài 4 (tự viết lại):** viết `even_xor(nums)` trả XOR của các số chẵn (không có số chẵn -> `0`).

**Bài 5 (mini challenge):** viết `even_and(nums)` trả AND của các số chẵn, và trả `0` nếu không có số chẵn nào.

In [24]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().evenNumberBitwiseORs([3, 5, 12, 10]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def odd_or_todo(nums):
    result = 0
    # TODO: với mỗi x lẻ (x % 2 == 1) thì result |= x
    return result

14


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: chỉ xét `12` và `10`. Bài 2: `20 = 10100`, `6 = 00110`.

**Mức 2 (vừa):** Bài 3: đổi điều kiện lọc thành `x % 2 == 1`. Bài 4: `result ^= x` với `x` chẵn. Bài 5: khởi tạo AND bằng `None` (hoặc `-1`), cập nhật `&=`, cuối cùng đổi `None` thành `0`.

**Mức 3 (gần đáp án):** Bài 1: `12 = 1100`, `10 = 1010` -> OR `1110 = 14`. Bài 2: `10100 | 00110 = 10110 = 22`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
assert Solution().evenNumberBitwiseORs([3, 5, 12, 10]) == 14

# Bài 2
assert 0b10100 | 0b00110 == 0b10110 == 22
assert Solution().evenNumberBitwiseORs([20, 6, 9]) == 22

# Bài 3
def odd_or(nums):
    result = 0
    for x in nums:
        if x % 2 == 1:
            result |= x
    return result
assert odd_or([1, 2, 3, 4, 5, 6]) == 7
assert odd_or([2, 4]) == 0

# Bài 4
def even_xor(nums):
    result = 0
    for x in nums:
        if x % 2 == 0:
            result ^= x
    return result
assert even_xor([2, 4, 6]) == 0
assert even_xor([1, 8, 16]) == 24
assert even_xor([7, 9]) == 0

# Bài 5
def even_and(nums):
    result = None
    for x in nums:
        if x % 2 == 0:
            result = x if result is None else result & x
    return 0 if result is None else result
assert even_and([6, 12, 3]) == 4          # 0110 & 1100 = 0100
assert even_and([7, 9, 11]) == 0
assert even_and([8]) == 8
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "gộp bit của các phần tử thỏa điều kiện":**

```python
result = 0                      # OR / XOR: 0 ; AND: -1 (toàn bit 1)
for x in nums:
    if x % 2 == 0:              # điều kiện lọc
        result |= x             # hoặc ^= , &=
return result
```

**Checklist trước khi submit:**

- [ ] Chỉ OR các số **chẵn** (đã lọc `x % 2 == 0`)?
- [ ] Dùng toán tử **bit** `|` (không phải `or`, không phải `+`)?
- [ ] Khởi tạo `0` (không phải `nums[0]`)?
- [ ] Không có số chẵn thì trả `0`?
- [ ] Đã thử `[1,2,3,4,5,6]`, `[7,9,11]`, `[1,8,16]`, `[100]`, một phần tử?
- [ ] Chữ ký đúng `evenNumberBitwiseORs(self, nums: List[int]) -> int`?